# Query Generation Agent

In [111]:
import os
import json
from typing import List, Tuple, Dict
from dotenv import load_dotenv
from openai import OpenAI

In [71]:
load_dotenv()

True

In [72]:
system_message = """
    You specialize in crafting best and most efficient google search operators that work well.
    The prompt you generate will be used as queries for Exa and SerpAPI web search later on.
""".strip()

## SerpAPI 

In [92]:
def build_serpapi_user_prompt() -> str:
    """
    Build a dynamic user message for LLM to generate search operators that fit for SerpAPI usecase
    """


    num_queries_serp = 10
    
    return f"""
            Generate search queries for my job search requirements.

            Requirements:
            - I need a remote AI engineering role focused on agentic systems, AI agents, RAG pipelines, LoRA/QLoRA fine-tuning, AI integration, AI-driven applications, and LLM engineering including frontier and open-source Hugging Face models.
            - The role MUST be a remote role available for applicants from the global/worldwide region or ASIA, APAC or any upper regional category that includes South Korea. (Generate most used queries as possible that can include Asia region)
            - Junior, mid-level, internship are preferred. Senior roles are still acceptable if realistic for a 3-year-experience developer.
            - {num_queries_serp} queries must be generated, formatted in the google search operator style.
            - It must avoid general job boards like LinkedIn, Indeed.com, ziprecruiter.com etc which have none to least global remote job offers. It must be queries that search for companies' direct hiring page or ATS including GreenHouse, Lever, Workable, Ashby, Recuitee, Breezy HR, Zoho Recruit, jobs.smartrecruiters or anything similar to that (ATS) or tech startup job boards that include global remote jobs offers. 
            - In your queries, include strong job-intent keywords commonly used by companies to focus on REAL JOBS (e.g. "careers", "job openings", "open roles", "jobs", or anything you think is great to get valid search results on job openings.) so that it avoids irrelevant results like blogs, articles, forums, or news pages.
            - Generate search queries in these formats:
              for example, site:boards.greenhouse.io ("AI engineer" OR "LLM" OR "RAG" OR "LangChain" OR "LangGraph" OR "agent") ("APAC" OR "Asia" OR "global" OR "worldwide" OR "timezone overlap") -senior
              (it's jsut an example for your strcutural reference, but you can use your creativity to generate many random search queries that work the best!!)
              IMPORTANT: it has to target one of ATS sites, role title and/or skill names, remote regions(worldwide, global, work from anywhere, or Asia etc), and excluding any words like "senior", "lead", "head", "staff engineer" or anything that implies a senior role. 
 
            - Output exactly this JSON schema (List[str] in JSON):

            [
                "... exactly {num_queries_serp} strings ..."
            ]
            

            Format Rules:
            - Return only raw JSON.
            - No markdown fences.
            - No explanation.
            - Each list must contain exactly {num_queries_serp} strings.
            """.strip()

In [93]:
llm = OpenAI()

In [94]:
serp_query_results = llm.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": build_serpapi_user_prompt()}
    ]
)

In [98]:
import json

data = json.loads(serp_query_results.choices[0].message.content)

for q in data:
    print(q)

site:boards.greenhouse.io ("AI engineer" OR "LLM engineer" OR "AI agent" OR "agentic" OR "RAG" OR "LangChain" OR "LangGraph") ("remote" OR "work from anywhere" OR "global" OR "worldwide" OR "APAC" OR "Asia" OR "South Korea") ("jobs" OR "careers" OR "open roles" OR "job openings") -senior -lead -head -staff -principal -director -manager -sr
site:jobs.lever.co ("LLM" OR "LLM engineer" OR "RAG" OR "retrieval" OR "LangChain" OR "agent" OR "AI agents" OR "agentic systems") ("remote" OR "worldwide" OR "global" OR "APAC" OR "Asia" OR "Korea" OR "South Korea") ("jobs" OR "careers" OR "open roles" OR "job openings") -senior -lead -head -staff -principal -director -manager -sr
site:jobs.workable.com ("LoRA" OR "QLoRA" OR "LoRA fine-tuning" OR "QLoRA" OR "fine-tuning" OR "fine tuning") ("Hugging Face" OR "transformers" OR "open-source models" OR "HF") ("remote" OR "work from anywhere" OR "global" OR "Asia" OR "APAC" OR "South Korea") ("jobs" OR "careers" OR "open roles" OR "internship" OR "intern

## Exa

In [77]:
def build_exa_user_prompt() -> str:
    """
    Build a dynamic user message for Exa semantic search queries
    """

    num_queries_exa = 10

    return f"""
            Generate semantic search queries for my job search.

            Requirements:
            - I need a remote AI engineering role focused on agentic systems, AI agents, RAG pipelines, LoRA/QLoRA fine-tuning, AI integration, AI-driven applications, and LLM engineering including frontier and open-source Hugging Face models.
            - The role MUST be remote and available globally, worldwide, or in regions including ASIA/APAC (including South Korea).
            - Junior, mid-level, internship are preferred. Senior roles are acceptable if realistic for a ~3-year-experience developer.

            - Queries should be written in natural, human-like language optimized for semantic search (NOT heavy Google operator style).
            - Do NOT focus only on ATS domains — allow broader discovery including company pages, startup hiring pages, and lesser-known sources.

            - Include strong job-intent signals in natural form:
              (e.g. "hiring", "job opening", "career opportunity", "looking for", "open role")

           - Avoid irrelevant content such as blogs, articles, tutorials, forums, and news:
              Instead of using negative phrases, prioritize strong positive job-intent signals like:
              "hiring", "job opening", "open role", "career opportunity", "we are hiring"

            - IMPORTANT: Do NOT over-constrain queries.
              Keep each query simple and realistic (1–2 core ideas).
              Spread variations across multiple queries instead of stacking conditions.

            - Prioritize recall over strict filtering:
              queries should help discover jobs that are harder to find via traditional search.

            - Output exactly this JSON schema:

            [
                "... exactly {num_queries_exa} strings ..."
            ]
            

            Format Rules:
            - Return only raw JSON.
            - No markdown fences.
            - No explanation.
            - The list must contain exactly {num_queries_exa} strings.
            """.strip()

In [78]:
exa_query_results = llm.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": build_exa_user_prompt()}
    ]
)

In [79]:
print(exa_query_results.choices[0].message.content)

[
  "Hiring remote AI engineer for agentic systems and AI agents, open to junior or mid-level global applicants",
  "Remote job opening in LLM engineering and RAG pipelines, worldwide hiring for junior and mid roles",
  "We are hiring a remote engineer for LoRA/QLoRA fine-tuning and Hugging Face model work, open role for global candidates",
  "Career opportunity: remote AI integration engineer building AI-driven applications, welcoming applicants worldwide and APAC",
  "Looking for a remote AI engineer internship focused on agentic systems and RAG pipelines, open to applicants in South Korea and APAC",
  "Open role: remote LLM engineer working with open-source models and Hugging Face, hiring junior to mid-level worldwide",
  "Hiring remotely: engineer for AI agents and autonomous workflows, career opportunity for 0-3 years experience, global",
  "Remote job opening: fine-tuning with LoRA/QLoRA and building RAG pipelines, hiring worldwide including Asia/APAC",
  "We are hiring a remote 

## Parsing Json

In [109]:
def process_response() -> Dict[str,List[str]]:
    """
    load the LLM JSON response and separate each key (ser, exa) for easier process on SerpAPI and Exa individually.
    """

    try:
        serp_queries = json.loads(serp_query_results.choices[0].message.content)

        if not isinstance(serp_queries, list): 
            raise ValueError("Serp queries must be in a list")
        
        if not all(isinstance(serp_query, str) for serp_query in serp_queries):
            raise ValueError("Serp queries must be strings")
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse LLM JSON response for SerpAPI: {e}\n")

    try:
        exa_queries = json.loads(exa_query_results.choices[0].message.content) 

        if not isinstance(exa_queries, list):
            raise ValueError("Exa queries must be in a list")
        
        if not all(isinstance(exa_query, str) for exa_query in exa_queries):
            raise ValueError("Exa queries must be strings")
    except json.JSONDecodeError as e:
        raise ValueError(f"Failed to parse LLM JSON response for Exa: {e}\n")

    if len(serp_queries) != 10 or len(exa_queries) != 10:
        raise ValueError(
            f"Expected exactly 10 serp and 10 exa queries, got serp={len(serp_queries)}, exa={len(exa_queries)}."
        )

    return {
        "serp": serp_queries,
        "exa": exa_queries
    }

In [110]:
process_response()

{'serp': ['site:boards.greenhouse.io ("AI engineer" OR "LLM engineer" OR "AI agent" OR "agentic" OR "RAG" OR "LangChain" OR "LangGraph") ("remote" OR "work from anywhere" OR "global" OR "worldwide" OR "APAC" OR "Asia" OR "South Korea") ("jobs" OR "careers" OR "open roles" OR "job openings") -senior -lead -head -staff -principal -director -manager -sr',
  'site:jobs.lever.co ("LLM" OR "LLM engineer" OR "RAG" OR "retrieval" OR "LangChain" OR "agent" OR "AI agents" OR "agentic systems") ("remote" OR "worldwide" OR "global" OR "APAC" OR "Asia" OR "Korea" OR "South Korea") ("jobs" OR "careers" OR "open roles" OR "job openings") -senior -lead -head -staff -principal -director -manager -sr',
  'site:jobs.workable.com ("LoRA" OR "QLoRA" OR "LoRA fine-tuning" OR "QLoRA" OR "fine-tuning" OR "fine tuning") ("Hugging Face" OR "transformers" OR "open-source models" OR "HF") ("remote" OR "work from anywhere" OR "global" OR "Asia" OR "APAC" OR "South Korea") ("jobs" OR "careers" OR "open roles" OR "i